# Topic: LLM for Recommendation Systems
---

Version: Aug 2025 Written by Jin Yuze (jin.yuze@u.nus.edu)

###################

Under Construction:

Progress:
draft for detailed discussion is done,
working on code examples, and better explanations.

###################

---
## Introduction

In this topic, we want to explore the cutting edge of using Large Language Models (LLMs) for recommendation systems. LLMs have shown great potential in understanding and generating human-like text, and they can be leveraged to enhance the recommendation process by providing more personalized and context-aware suggestions.

In recent years, there has been a surge of interest in using LLMs for recommendation systems. These models can analyze user preferences, item descriptions, and contextual information to generate recommendations that are more aligned with user interests. The models can take the textual data associated with items (such as product descriptions, reviews, and user-generated content) and use it to better understand the relationships between users and items, leading to more accurate and relevant recommendations, compare to conventional collaborative filtering or content-based methods.

In this topic, we will explore this area following a reseach paper that proposes a noval approach.

The code is modified from project `LLM4REC`

[LLM-Enhanced User-Item Interactions: Leveraging Edge Information for Optimized Recommendations](https://arxiv.org/abs/2402.09617)

[Code Repo](https://github.com/anord-wang/LLM4REC)

#### Prelude

We assume you already have some basic knowledges on: 

- Conventional recommendation systems, such as collaborative filtering and content-based filtering, which are the already covered content in this course.
- Basic knowledge of Large Language Models (LLMs), such as Natural Language Processing (NLP) tasks, and how LLMs can be used to process and generate text, Attention Mechanism, Transformers, etc. (We have covered these basics in the course as well).
- Basic knowledge of PyTorch, such as how to define a model, how to train a model, and how to use PyTorch's built-in functions for optimization and loss calculation. 

#### Example Code
Since the whole code repo is quite complex, we will not go through all the components in this notebook. Instead, here we only show the main learning points of the topic. 

We provide a full version of the code repo as additional materials, which you are recommanded to try out by yourself.
We have provided a detailed instruction on how to run the code, and we provide the checkpoints for you to test the code without training the model from scratch (which will take some GPU resouces and long time).
And in the code repo, we provide detailed comments on each component, and a more technical explanation of the model architecture and training process.

---
## Overview

It is not trivial to use LLMs for recommendation systems, as the two systems have different assumptions on data structure. 
The paper we are going to explore proposes a novel approach that tries to fill in the gap between graph-based user & item interactions and LLMs' text-based understanding.

In previous studies in our module, we have seen the conventional recommendation methods, such as collaborative filtering and content-based filtering, which are based on user-item interactions. 
These methods typically rely on the assumption that user-item interactions can be represented as a bipartite graph, where users and items are nodes, and interactions (such as ratings or clicks) are edges. 
We have also seen how graph neural networks (GNNs) can be used to model these interactions, allowing for the aggregation of information from neighboring nodes to improve recommendations. 
These methods are shown to be effective in capturing the relationships between users and items.

While you have seen how LLMs can be used for various NLP tasks, such as text generation, sentiment analysis, and question answering, they are typically trained on large text corpora and are not directly designed to handle graph-based data. 
LLMs excel at understanding and generating text, but they do not inherently understand the structure of user-item interactions as a graph.

Thus how to bridge the gap between these two paradigms is a key challenge in using LLMs for recommendation systems.

---
### Idea

In this example code, we will explore one solution to this challenge.

The idea of this example paper is to leverage the edge information in user-item interactions to enhance the recommendations generated by LLMs.
The authors propose a novel approach that combines the strengths of both graph-based methods and LLMs, allowing for more effective recommendations:

- **Graph-aware Attention Mechanism** – The authors modify the LLM’s attention module by incorporating additional bias terms derived from the graph structure of the user-item interactions. This includes both direct connections (user-item links) and indirect relationships (shortest paths in the interaction graph). By integrating these structural signals into the attention computation, the model can focus more on semantically or structurally related nodes, rather than relying purely on sequential token relationships.

- **Tokenizer and Embedding Modifications** – To handle user and item IDs without losing their semantic identity, the authors adjust the tokenizer so that these IDs are treated as indivisible tokens. Special embeddings are then assigned to these tokens, enabling the LLM to recognize and represent specific users and items consistently during training and inference.

- **Prompt Design** – The input to the LLM is carefully constructed to include textual descriptions, user profiles, and the graph-based interaction information. This design ensures the model receives both semantic content and structural context, improving its ability to generate relevant recommendations.

- **Two-stage Training Strategy** – The training process consists of a pretraining stage, where the model learns general patterns from the full user-item corpus using a text generation objective, and a fine-tuning stage, where the model is optimized for the recommendation task by predicting the most likely items for a given user prompt.

### Terminology and Problem Statement:

In our description, we will follow the paper's terminology and problem statement, which we copy from the paper: 

> Consider the existence of `I` users and `J` items, let `X`<sub>`ij`</sub> be the binary interaction (e.g., purchase) matrix between user `i` and item `j`. 
> Besides, we collect user descriptions, item descriptions (e.g., prices, brand, category, title), user reviews for items, and explanations of user purchase reasons.
> We denote `T`<sub>`i`</sub> as the descriptions of the user `i`, `T`<sub>`j`</sub> as the descriptions of the item `j`, `T`<sub>`ij`</sub> as the joint texts of the user `i` and item `j`, such as user reviews and purchase reasons for items. 
> We unify all textual descriptions into `T` that includes `N` sequences, `k` indexes the tokens in each sequence, and `T`<sub>`nk`</sub> is the `𝑘`-th token in the `n`-th sequence. 
> Our goal is to leverage LLMs and graphs to develop a generative recommender system that takes a prompt, including a user ID and a user’s historical interaction records with items, and generates product recommendations to the user.

---

### Step-1 Attention may not be enough

Let's start from looking at our data. 

In our data, we have the following information:
- User's profile, Item's description, User's reviews. These are textual data. LLM can learn from these data to understand the user's preferences and item characteristics.
- User-item interactions, and second-order relationships between items. These are graph-based data. We can use these data to enhance the LLM's understanding of the relationships between users and items.

For the textual data, it is straightforward to use LLMs to process them. We can simply feed the text into the LLM and let it generate embeddings or predictions based on the text.

For the graph-based data, we need to do some extra work to enable the LLM to process them.

Previous, Attention mechanism is the key component to capture the relationships between tokens in the **text**. However, when we provide information like: 
```
<user_123> has interaction with <item_abc>
```
The attention mechanism can only provide the "language" relationship between the tokens `<user_123>` and `<item_abc>`, while the actual graph-based relationship is missing, thus the model may not fully understand the relationship between the user and the item.

To address this, we need to modify the attention mechanism to incorporate the graph-based relationships.

The original attention is defined as: 
<math display="block">
<mi>A</mi><mi>t</mi><mi>t</mi><mo stretchy="false">(</mo><mi>Q</mi><mo>,</mo><mi>K</mi><mo>,</mo><mi>V</mi><mo stretchy="false">)</mo><mo>=</mo><mi>S</mi><mi>o</mi><mi>f</mi><mi>t</mi><mi>m</mi><mi>a</mi><mi>x</mi><mo stretchy="false">(</mo><mfrac><mrow><mi>Q</mi><msup><mi>K</mi><mi>T</mi></msup></mrow><msqrt><msub><mi>d</mi><mi>k</mi></msub></msqrt></mfrac><mo stretchy="false">)</mo><mi>V</mi>
</math>
(We will omit the detailed explanation of the attention mechanism here.)

As you can see, the attention is only defined based on the textural relationship between the tokens, thus, in the example paper, the Attention between `user` or `item` is modified to: 
<math display="block">
<mi>A</mi><mi>t</mi><msup><mi>t</mi><mo>′</mo></msup><mo stretchy="false">(</mo><mi>Q</mi><mo>,</mo><mi>K</mi><mo>,</mo><mi>V</mi><mo stretchy="false">)</mo><mo>=</mo><mi>S</mi><mi>o</mi><mi>f</mi><mi>t</mi><mi>m</mi><mi>a</mi><mi>x</mi><mo stretchy="false">(</mo><mfrac><mrow><mi>Q</mi><msup><mi>K</mi><mi>T</mi></msup></mrow><msqrt><msub><mi>d</mi><mi>k</mi></msub></msqrt></mfrac><mo>+</mo><mi>R</mi><mo stretchy="false">)</mo><mi>V</mi>
</math>

with an additional bias term `R` that is defined as: 
<math display="block">
<mi>R</mi><mo>=</mo><msup><mi>R</mi><mrow><mi>c</mi><mi>o</mi><mi>n</mi><mi>n</mi></mrow></msup><mo>+</mo><msup><mi>R</mi><mrow><mi>p</mi><mi>a</mi><mi>t</mi><mi>h</mi></mrow></msup>
</math>
where `R`<sub>`con`</sub> is the direct connection between the user and item, and `R`<sub>`path`</sub> is the second-order relationship between items.

For the direct connection, it is easily defined as: 
<math display="block"><msubsup><mi>R</mi><mrow><mi>i</mi><mi>j</mi></mrow><mrow><mrow><mi mathvariant="normal">c</mi><mi mathvariant="normal">o</mi><mi mathvariant="normal">n</mi><mi mathvariant="normal">n</mi></mrow></mrow></msubsup><mo>=</mo><mrow data-mjx-texclass="INNER"><mo data-mjx-texclass="OPEN">{</mo><mtable columnalign="left left" columnspacing="1em" rowspacing=".2em"><mtr><mtd><mn>1</mn><mo>,</mo></mtd><mtd><mrow><mtext>if there is a direct connection between node&nbsp;</mtext><mrow><mi>i</mi></mrow><mtext>&nbsp;and&nbsp;</mtext><mrow><mi>j</mi></mrow><mtext>,</mtext></mrow></mtd></mtr><mtr><mtd><mn>0</mn><mo>,</mo></mtd><mtd><mtext>otherwise.</mtext></mtd></mtr></mtable><mo data-mjx-texclass="CLOSE" fence="true" stretchy="true" symmetric="true"></mo></mrow></math>.

This represents the strong direct interaction between the user and the item.

While for the indirect connection, it is corresponding to the "collaborative information" in conventional solutions. It is defined as:
```
A normalized shortest path score between nodes, which is computed based on the entire graph.
```
The formula is: 
<math display="block"><msubsup><mi>R</mi><mrow><mi>i</mi><mi>j</mi></mrow><mrow><mi>p</mi><mi>a</mi><mi>t</mi><mi>h</mi></mrow></msubsup><mo>=</mo><mn>1</mn><mo>−</mo><mfrac><mrow><mi>δ</mi><msub><mi>P</mi><mrow><mi>i</mi><mi>j</mi></mrow></msub></mrow><mrow><mi>m</mi><mi>a</mi><mi>x</mi><mo stretchy="false">(</mo><mi>P</mi><mo stretchy="false">)</mo></mrow></mfrac></math>
Where `P` is the length of the shortest path between nodes `i` and `j`. and `δ` is a factor between 0 and 1. 

With this modified Attention mechanism, now the model can capture both the textual relationships and the graph-based relationships between users and items.

TODO: Example code

---

### Step-2 Take care of the Tokenizer

In LLM4Rec, the tokenizer of the base LLM (e.g. GPT-2) is modified to handle user IDs and item IDs as special whole tokens. 

This is necessary because user/item IDs are not normal words: they are unique identifiers that should not be broken into subwords. 
If we use an unmodified tokenizer on an ID like `user_123`, it would be split into pieces (e.g. `user`, `_`, and `123`). 
Such splitting causes the model to lose the identity of the user or item; the token `user` or `123` alone has no specific meaning for a particular user 123. 

To fix this, LLM4Rec customizes the tokenizer vocabulary so that each user and item ID is treated as an indivisible token. 
In practice, all strings like user_<id> and item_<id> are added as single tokens. 

The model is then given special embeddings for these new tokens. 
This means the LLM will learn a distinct embedding vector for each user and each item, allowing it to recognize specific users/items consistently during training and inference. 
By preserving each ID as one token, we ensure the model can capture the unique semantics of, say, user_123 (distinct from user_456, etc.) instead of seeing them as generic words or numbers. 

Overall, modifying the tokenizer in this way injects the traditional ID-based collaborative filtering signals into the LLM’s vocabulary, bridging the gap between ID-based representations and natural language text.


---

### Step-3 Prompt Design

With the tokenizer set up to handle ID tokens, the next step is to design the input prompts that will feed the model during training (and later, for recommendations). 

According to the problem definition, the LLM’s input prompt is constructed to include a user’s ID and that user’s historical interaction records with items. 
In other words, each training sample is turned into a structured textual prompt that provides the model with both who the user is and what they have interacted with, along with descriptive context. 

The prompt typically contains several components:
- **User ID token** – e.g. `user_42`. This special token identifies which user we are talking about and lets the model retrieve that user’s embedding.

- **User profile description** (if available) – textual information about the user’s characteristics or preferences. This is natural language (normal vocabulary) that gives semantic context about the user. For example, if we know `user_42` is “a fan of science fiction and technology,” that might be included in the prompt as a sentence.

- **Interaction history** (graph-based info) – a list of item IDs that the user has interacted with, represented by their special item tokens. These are often incorporated into a sentence using a template phrase to indicate the relationship. For instance, we might write: `user_42` has interacted with `item_5`, `item_17`, `item_36` etc., where each `item_x` is a token for an item the user engaged with. This part explicitly injects the user-item graph structure (which items the user connected with) into the prompt in a readable way.

- **Item descriptions** – for each item in the history, or for certain key items, the prompt can include brief natural-language descriptions (e.g. item titles, categories, or attributes). These are “hard” (normal text) tokens that provide semantic content about the item. By reading item descriptions, the LLM can understand what the items are (for example, that `item_5` is “a popular sci-fi novel” and `item_17` is “a blockbuster action movie”, etc.), rather than treating them as opaque IDs. This helps the model connect collaborative signals with content-based semantics.

Using these components, we construct a single combined prompt string for each training example. An example prompt structure could be:
```
<user_42> (a sci-fi fan) has interacted with <item_5> (a science fiction novel), <item_17> (an action movie), <item_36> (a tech gadget), <user_42> will interact with ...
```

In this example, the prompt first gives the user ID and a quick profile note in parentheses, then lists item IDs the user has interacted with, each followed by a short description, and finally ends with a phrase like “... will interact with” to cue the model for prediction. 

The mix of “soft” tokens (IDs) and “hard” tokens (natural language) provides rich context to the LLM. 
The user and item ID tokens inject the collaborative filtering signal (who the user is and which items they connected to), while the descriptive text around them ensures the model also grasps the semantic content. 

This prompt design is crucial: it ensures the model sees both semantic content and structural context in one sequence. 
By reading a prompt, the LLM can understand the user’s preferences (from profile and past item descriptions) and the user’s interaction graph (from the ID tokens and the “has interacted with” relations), which together should improve its ability to generate relevant recommendations.

---
### Step-4 Pre-training and Fine-tuning

LLM4Rec employs a two-stage training strategy: a pretraining stage followed by a fine-tuning stage. This approach allows the model to first learn general patterns from the data and then specialize in making accurate recommendations.

#### Pretraining Stage

In this stage, the augmented LLM is trained on a large corpus derived from all users’ interactions and content in a generative manner. 

Essentially, we use a text-generation (language modeling) objective over the assembled prompts and texts. 
The model reads prompts like the ones described above and tries to predict the next tokens in the sequence, just as it would in normal language modeling. 
Through this process, the LLM learns the general structure of the data, including how user and item tokens typically co-occur and relate, and how descriptions are phrased. 

For example, it will learn patterns such as `<user_X> has interacted with <item_Y> ...` and get used to seeing certain item types following certain users. 
Importantly, because the pretraining corpus includes both the collaborative information (IDs and their relationships) and content information (descriptions, reviews, etc.), the model starts to capture both types of signals.

The goal here is not yet to make final recommendations, but to familiarize the model with the “language” of our recommendation scenario. 
By the end of pretraining, the LLM has a good grasp of how to represent users and items in the shared embedding space and how to generate relevant text given a mix of IDs and words. 
It has essentially learned general patterns of user-item interactions and content from the data.

#### Fine-Tuning Stage

After pretraining, the model is already fluent in our recommendation-oriented language, but we want it to excel at the specific task of recommending the correct items. 

In fine-tuning, the training objective is shifted to predicting the most likely item(s) for a given user prompt, rather than free-form text generation. 
The model is presented with prompts based on each user’s historical interactions (often with some interactions held out or masked) and is trained to output the missing/next item tokens that the user would interact with.

In other words, we optimize the LLM to minimize recommendation errors. It should produce the actual items the user would like, instead of just producing plausible sentences. 
This can be implemented by adding a prediction head or by treating the next item as the target in the language model, but the key is that the loss function directly measures recommendation accuracy (e.g. likelihood of the true item) rather than plain language perplexity. 

For instance, given a prompt `<user_42> has interacted with <item_5>, <item_17>, <item_36>, <user_42> will interact with …`, the fine-tuning training will adjust the model’s weights so that it assigns high probability to the correct next item(s) that user 42 actually interacted with (and low probability to irrelevant items). 

During this stage, no new information is introduced; we’re using the same user-item data but focusing the model on making predictions. 
As the authors highlight, the fine-tuning objective is specifically crafted so that the model becomes “good at the recommendation task,” not just generating text that sounds correct.

Through the two-stage process, the LLM is aligned with the recommendation task. 
Pretraining on the full user-item corpus gives it a strong foundation in understanding users, items, and their textual contexts in a unified way. 
Fine-tuning then hones the model’s ability to generate the right item recommendations when given a user’s prompt. 

In summary, the model first learns how to speak the language of the data, and then learns how to speak the language of recommendations. 
By the end of training, when you provide a prompt with a particular user ID and history, the LLM can effectively leverage both the collaborative signals and content knowledge it acquired to output likely items that the user will be interested in – fulfilling the goal of a generative recommender system.

---
### Discussion

LLM4Rec is one example of a growing family of methods that integrate LLMs into recommender systems. 

Broadly, existing work in this space follows several directions:

- Prompt-based zero/few-shot approaches, where the LLM is kept frozen and clever prompt engineering is used to elicit recommendations without retraining.

- Feature-augmentation approaches, where the LLM generates additional descriptive or contextual features that are then used by a separate recommender.

- End-to-end fine-tuned approaches, such as LLM4Rec, that modify the model’s architecture or vocabulary to directly learn from collaborative and content data together.

- Hybrid models with graphs or retrieval, where LLMs are paired with external modules like GNNs, LoRA, or RAG for better scalability and structure awareness.

While these methods differ in complexity and cost, they share the goal of leveraging the LLM’s semantic understanding while still capturing user–item relationships effectively.

---
### Outlook

The use of LLMs in recommendation is still in its early stages but holds significant promise, especially for tasks requiring rich semantic context, cold-start handling, or explanation generation. 

Future progress will likely focus on improving scalability (handling millions of users/items), enhancing alignment with real-world ranking metrics, and combining structured reasoning over graphs with the LLM’s generative abilities. 

For students and researchers, this area offers an opportunity to work at the intersection of natural language processing, graph learning, and recommender systems—tackling open questions such as how to efficiently integrate large-scale interaction graphs into LLM reasoning while keeping the system fast, accurate, and adaptable.